In [17]:
# Ensure datasets library is installed and load the dataset as in EDA notebook
%pip install datasets -q
from datasets import load_dataset

dataset = load_dataset("millat/e-commerce-orders")
df = dataset['train'].to_pandas()
df.head()

Note: you may need to restart the kernel to use updated packages.


,order_id,customer_id,product_id,category,price,quantity,order_date,shipping_date,delivery_status,payment_method,device_type,channel,shipping_address,billing_address,customer_segment
0,b8ec9f86-5919-4b71-a5f7-945e7c0a3db0,2031,845,Books,45.95,4,2024-04-20 14:59:58.897063,2024-04-27 14:59:58.897063,Shipped,PayPal,Mobile,Paid Search,"976 Kevin Station, Davidmouth, Hawaii 92563","38182 Ariel Expressway, Campbellland, Oklahoma...",VIP
1,5ea92c47-c5b2-4bdd-8a50-d77efd77ec89,2350,995,Electronics,403.17,3,2024-04-20 14:59:58.897063,2024-04-22 14:59:58.897063,Delivered,PayPal,Mobile,Paid Search,"72166 Cunningham Crescent, East Nicholasside, ...","38199 Edwin Plain, Johnborough, Maine 81826",Returning
2,5cc48ce0-2c6d-4448-af3f-96f8a910d45b,1818,997,Beauty,317.45,2,2024-04-20 14:59:58.897063,2024-04-27 14:59:58.897063,Shipped,Credit Card,Mobile,Email,"2446 Johnson Junctions, Lynchtown, Minnesota 7...","58086 Smith Stream Suite 994, Lake Pamelabury,...",Returning
3,74d5c0f4-53f0-4367-a5c5-baaa114c2d9f,472,385,Home,24.08,3,2024-04-20 14:59:58.897063,2024-04-24 14:59:58.897063,Shipped,PayPal,Tablet,Social,"3113 Jessica Knolls, North Joshuafort, Alabama...","484 Palmer Harbors Apt. 866, Dustintown, Nebra...",VIP
4,7a630323-8ac8-406e-875a-4bcdead440ab,1075,31,Clothing,494.90,1,2024-04-20 14:59:58.897063,2024-04-25 14:59:58.897063,Delivered,PayPal,Tablet,Organic,"58196 Burgess Heights Suite 315, Douglasland, ...","67094 Schaefer Villages Suite 369, Douglasches...",VIP


# 4. MLOps Deployment: MLflow, GCS, and Vertex AI
This notebook documents the transfer of model artifacts to Google Cloud Storage and the deployment of the model as a live endpoint on Vertex AI.

## 1. Import Required Libraries
We import libraries for model packaging, cloud storage, and deployment.

In [18]:
import joblib
import os
# For Google Cloud Storage and Vertex AI, use the google-cloud-storage and google-cloud-aiplatform packages
# !pip install google-cloud-storage google-cloud-aiplatform
from google.cloud import storage
from google.cloud import aiplatform

## 2. Package and Save the Final Model
We combine the preprocessing pipeline and the best classification model into a single pipeline and save it as model.joblib.

In [19]:
# Load the best model pipeline from previous training (if not already in memory)
model_path = '../artifacts/model.joblib'

if os.path.exists(model_path):
	cls_pipeline = joblib.load(model_path)
else:
	# Example: create a dummy pipeline for demonstration (replace with your actual pipeline)
	from sklearn.pipeline import Pipeline
	from sklearn.preprocessing import StandardScaler
	from sklearn.ensemble import RandomForestClassifier

	# Assume 'customer_segment' is the target
	X = df.drop(columns=['customer_segment'])
	y = df['customer_segment']

	# For demonstration, use only numeric columns
	numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
	X_numeric = X[numeric_cols]

	cls_pipeline = Pipeline([
		('scaler', StandardScaler()),
		('clf', RandomForestClassifier(n_estimators=10, random_state=42))
	])
	cls_pipeline.fit(X_numeric, y)

# Save the model pipeline (optional, if you want to overwrite)
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(cls_pipeline, model_path)
print('Model pipeline saved to ../artifacts/model.joblib')

Model pipeline saved to ../artifacts/model.joblib


## 3. Prepare requirements.txt
We list all required Python packages for deployment.

## 4. Upload Artifacts to Google Cloud Storage (GCS)
We upload model.joblib and requirements.txt to a GCS bucket for deployment.

In [22]:
from google.oauth2 import service_account
from google.auth.exceptions import DefaultCredentialsError
import google.auth

# Set your GCS bucket name
bucket_name = 'ml-final-project-himantha-team-aurex'
model_artifact_dir = 'models/segment_classifier'
model_artifact_uri = f'gs://{bucket_name}/{model_artifact_dir}'

# Use the notebook variable if it exists; otherwise fall back to the expected path
sa_key_path = globals().get('sa_key_path', '../artifacts/service-account.json')
requirements_path = '../artifacts/requirements.txt'

# Create a requirements file if it does not already exist
os.makedirs('../artifacts', exist_ok=True)
if not os.path.exists(requirements_path):
    with open(requirements_path, 'w') as f:
        f.write('scikit-learn\npandas\njoblib\ngoogle-cloud-storage\ngoogle-cloud-aiplatform\n')

credentials = None
gcs_project = None

# Initialize GCS client (service-account file first, then ADC fallback)
if os.path.exists(sa_key_path):
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_key_path
    credentials = service_account.Credentials.from_service_account_file(sa_key_path)
    gcs_project = credentials.project_id
    print(f'Using service account key: {sa_key_path}')
else:
    try:
        credentials, detected_project = google.auth.default()
        gcs_project = globals().get('project', detected_project) or detected_project
        print('Service account key not found. Using Application Default Credentials from gcloud login.')
    except DefaultCredentialsError:
        print('No credentials found for GCS upload.')
        print('Run this once in terminal: gcloud auth application-default login')
        print('Then re-run this cell to upload artifacts.')

if credentials is not None:
    client = storage.Client(project=gcs_project, credentials=credentials)
    bucket = client.lookup_bucket(bucket_name)

    if bucket is None:
        bucket_location = globals().get('location', 'us-central1')
        bucket = client.bucket(bucket_name)
        bucket = client.create_bucket(bucket, location=bucket_location)
        print(f'Created bucket: {bucket_name} in {bucket_location}')
    else:
        print(f'Using existing bucket: {bucket_name}')

    # Upload model.joblib into a dedicated Vertex artifact directory
    model_blob = bucket.blob(f'{model_artifact_dir}/model.joblib')
    model_blob.upload_from_filename(model_path)
    print(f'Uploaded model.joblib to {model_artifact_uri}/model.joblib')

    # Upload requirements.txt into the same model artifact directory
    req_blob = bucket.blob(f'{model_artifact_dir}/requirements.txt')
    req_blob.upload_from_filename(requirements_path)
    print(f'Uploaded requirements.txt to {model_artifact_uri}/requirements.txt')

    # Expose URI for deployment cell
    MODEL_ARTIFACT_URI = model_artifact_uri
    print(f'MODEL_ARTIFACT_URI set to: {MODEL_ARTIFACT_URI}')

Service account key not found. Using Application Default Credentials from gcloud login.
Using existing bucket: ml-final-project-himantha-team-aurex
Uploaded model.joblib to gs://ml-final-project-himantha-team-aurex/models/segment_classifier/model.joblib
Uploaded requirements.txt to gs://ml-final-project-himantha-team-aurex/models/segment_classifier/requirements.txt
MODEL_ARTIFACT_URI set to: gs://ml-final-project-himantha-team-aurex/models/segment_classifier


## 5. Deploy Model to Vertex AI Endpoint
We import the model from GCS, select the pre-built Scikit-learn container, and deploy as a live endpoint.

In [23]:
# Set your project and region
project = 'ml-final-project-team-aurex'
location = 'us-central1'

# Fast mode: submit deploy job quickly and return, then wait only when needed
ASYNC_DEPLOY = True
DEPLOY_MACHINE_TYPE = 'n1-standard-2'
ENDPOINT_DISPLAY_NAME = 'auracart-segment-endpoint'
MODEL_DISPLAY_NAME = 'AuraCart Customer Segment Model'

# Prefer dedicated model artifact directory if defined in previous cell
MODEL_ARTIFACT_URI = globals().get('MODEL_ARTIFACT_URI', f'gs://{bucket_name}/models/segment_classifier')

# Initialize Vertex AI
aiplatform.init(project=project, location=location)

print('Checking for existing model...')
existing_models = aiplatform.Model.list(
    filter=f'display_name="{MODEL_DISPLAY_NAME}"',
    order_by='update_time desc'
 )

if existing_models:
    model = existing_models[0]
    print(f'Using existing model: {model.resource_name}')
else:
    print('Uploading model to Vertex AI...')
    print(f'Artifact URI: {MODEL_ARTIFACT_URI}')
    model = aiplatform.Model.upload(
        display_name=MODEL_DISPLAY_NAME,
        artifact_uri=MODEL_ARTIFACT_URI,
        serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest',
    )
    if not ASYNC_DEPLOY:
        model.wait()
    print(f'Model uploaded: {model.resource_name}')

print('\nChecking for existing endpoint...')
existing_endpoints = aiplatform.Endpoint.list(
    filter=f'display_name="{ENDPOINT_DISPLAY_NAME}"',
    order_by='update_time desc'
 )

if existing_endpoints:
    endpoint = existing_endpoints[0]
    print(f'Using existing endpoint: {endpoint.resource_name}')

    # If endpoint already has deployed models, skip redeploy to save time
    deployed_count = len(endpoint.gca_resource.deployed_models)
    if deployed_count > 0:
        print(f'Endpoint already has {deployed_count} deployed model(s). Skipping new deploy for fast startup.')
    else:
        print('Endpoint exists but has no deployed models. Deploying model now...')
        endpoint = model.deploy(
            endpoint=endpoint,
            machine_type=DEPLOY_MACHINE_TYPE,
            min_replica_count=1,
            max_replica_count=1,
            sync=not ASYNC_DEPLOY,
            deployed_model_display_name='auracart-segment-model',
        )
else:
    print('No existing endpoint found. Creating endpoint first...')
    endpoint = aiplatform.Endpoint.create(
        display_name=ENDPOINT_DISPLAY_NAME,
        sync=True
    )
    print(f'Created endpoint: {endpoint.resource_name}')

    print('Deploying model to new endpoint...')
    endpoint = model.deploy(
        endpoint=endpoint,
        machine_type=DEPLOY_MACHINE_TYPE,
        min_replica_count=1,
        max_replica_count=1,
        sync=not ASYNC_DEPLOY,
        deployed_model_display_name='auracart-segment-model',
    )

if ASYNC_DEPLOY:
    print('\nDeployment submitted in async mode (fast).')
    print('Run endpoint.wait() before prediction if deployment is still in progress.')
else:
    print('\nDeployment completed successfully.')
    print('Endpoint resource:', endpoint.resource_name)

print('\nNext quick checks if deployment fails:')
print('- Confirm Vertex AI endpoint quota for machine type in us-central1')
print('- Delete failed pending operations/endpoints in Vertex AI Console')
print('- Confirm artifact path contains model.joblib and requirements.txt')

Checking for existing model...
Using existing model: projects/1056586763320/locations/us-central1/models/4119490737765089280

Checking for existing endpoint...
No existing endpoint found. Creating endpoint first...
Creating Endpoint
Create Endpoint backing LRO: projects/1056586763320/locations/us-central1/endpoints/4688335722778722304/operations/8683258982892044288
Endpoint created. Resource name: projects/1056586763320/locations/us-central1/endpoints/4688335722778722304
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/1056586763320/locations/us-central1/endpoints/4688335722778722304')
Created endpoint: projects/1056586763320/locations/us-central1/endpoints/4688335722778722304
Deploying model to new endpoint...
Deploying model to Endpoint : projects/1056586763320/locations/us-central1/endpoints/4688335722778722304

Deployment submitted in async mode (fast).
Run endpoint.wait() before prediction if deployment is still in progress.

Next quick checks if d

Deploy Endpoint model backing LRO: projects/1056586763320/locations/us-central1/endpoints/4688335722778722304/operations/4516584902644203520


## 6. Test the Live Endpoint
Send a sample JSON payload to the endpoint and display the prediction response.

In [26]:
# 6) Test the live endpoint after deployment succeeds
import json
import numpy as np
import os

# Build one sample instance from the dataset used earlier
sample_df = df.drop(columns=['customer_segment'], errors='ignore').head(1).copy()

# Convert NaN -> None and NumPy scalars -> native Python types for JSON compatibility
instance = {}
for k, v in sample_df.iloc[0].to_dict().items():
    if v is None or (isinstance(v, float) and np.isnan(v)):
        instance[k] = None
    elif isinstance(v, np.generic):
        instance[k] = v.item()
    else:
        instance[k] = v

if 'endpoint' not in globals():
    print('Endpoint is not defined yet.')
    print('Run Cell 11 first.')
else:
    deployed_count = len(endpoint.gca_resource.deployed_models)
    if deployed_count == 0:
        print('Endpoint exists but no model is ready yet.')
        print('If you used async deploy, wait a few minutes and run this cell again.')
    else:
        print('Testing with endpoint:', endpoint.resource_name)
        print('Sample payload keys:', list(instance.keys())[:8], '...')

        try:
            prediction = endpoint.predict(instances=[instance])
            payload = {'instances': [instance]}
            response_obj = {'predictions': prediction.predictions}

            print('Request payload (JSON):')
            print(json.dumps(payload, indent=2, default=str))
            print('\nPrediction response (JSON):')
            print(json.dumps(response_obj, indent=2, default=str))

            os.makedirs('../artifacts', exist_ok=True)
            with open('../artifacts/prediction_request_response.json', 'w', encoding='utf-8') as f:
                json.dump({'request': payload, 'response': response_obj}, f, indent=2, default=str)
            print('\nSaved: ../artifacts/prediction_request_response.json')
        except Exception as e:
            print('Prediction call failed:')
            print(type(e).__name__, str(e))
            print('Likely reason: endpoint deployment is still in progress.')
            print('Wait a few minutes, then run this cell again.')

Endpoint exists but no model is ready yet.
If you used async deploy, wait a few minutes and run this cell again.
